# InsightForge AI — Agent 5: Insight Agent
Sends statistical and schema context to Gemini 1.5 Flash to generate executive insights and strategic recommendations.


In [ ]:
%pip install pandas google-generativeai
dbutils.library.restartPython()


In [ ]:
class InsightForgeState(TypedDict):
    """
    Shared state passed through all agents in the pipeline.
    Each agent reads what it needs and writes its output.
    No agent modifies another agent's output fields.

    Fields
    ------
    dataset_path     : path to the input CSV file
    gemini_key       : Gemini API key passed at runtime
    raw_df           : original DataFrame as uploaded
    cleaned_df       : DataFrame after cleaning agent runs
    schema_info      : column metadata detected by schema agent
    cleaning_report  : summary of all cleaning actions taken
    eda_results      : statistical analysis from EDA agent
    charts           : list of Plotly figure dicts from viz agent
    insights         : AI generated business insights text
    pdf_path         : path to the generated PDF report
    pipeline_log     : timestamped log of each agent execution
    errors           : list of error messages from any agent
    """
    dataset_path    : str
    gemini_key      : str
    raw_df          : Any
    cleaned_df      : Any
    schema_info     : dict
    cleaning_report : dict
    eda_results     : dict
    charts          : list
    insights        : str
    pdf_path        : str
    pipeline_log    : list
    errors          : list

print("✅ InsightForgeState defined")
print()
print("  State fields:")
fields = [
    ("dataset_path",     "input — path to CSV"),
    ("gemini_key",       "input — API key"),
    ("raw_df",           "Schema Agent reads this"),
    ("cleaned_df",       "Cleaning Agent writes this"),
    ("schema_info",      "Schema Agent writes this"),
    ("cleaning_report",  "Cleaning Agent writes this"),
    ("eda_results",      "EDA Agent writes this"),
    ("charts",           "Visualization Agent writes this"),
    ("insights",         "Insight Agent writes this"),
    ("pdf_path",         "Report Agent writes this"),
    ("pipeline_log",     "every agent appends to this"),
    ("errors",           "every agent appends on failure"),
]
for field, desc in fields:
    print(f"    {field:20} — {desc}")


In [ ]:
def log_event(state: InsightForgeState, agent: str, message: str) -> list:
    """
    Appends a timestamped log entry to the pipeline log.
    Called by every agent on start and completion.

    Parameters
    ----------
    state   : current pipeline state
    agent   : name of the calling agent
    message : what happened

    Returns
    -------
    list : updated pipeline log
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = f"[{timestamp}] {agent}: {message}"
    print(f"   {entry}")
    return state["pipeline_log"] + [entry]


def get_gemini_model() -> genai.GenerativeModel:
    """
    Returns a configured Gemini model instance.
    Re-reads the API key from widget each time to handle
    session restarts without needing to re-run setup cells.
    """
    key = dbutils.widgets.get("gemini_key")
    genai.configure(api_key=key)
    return genai.GenerativeModel(GEMINI_MODEL)


def safe_call_gemini(prompt: str, agent_name: str) -> str:
    """
    Wraps a Gemini API call with error handling.
    Cleans the response text to remove characters that
    fpdf2 cannot render with standard Helvetica font.

    Parameters
    ----------
    prompt     : the full prompt string to send
    agent_name : name of the calling agent for logging

    Returns
    -------
    str : cleaned response text or error message
    """
    try:
        m        = get_gemini_model()
        response = m.generate_content(prompt)
        text     = response.text

        # Remove characters unsupported by Helvetica in fpdf2
        replacements = {
            "\u2014": "-",    # em dash
            "\u2013": "-",    # en dash
            "\u2012": "-",    # figure dash
            "\u2011": "-",    # non-breaking hyphen
            "\u2010": "-",    # hyphen
            "\u2022": "-",    # bullet
            "\u2023": "-",    # triangle bullet
            "\u2043": "-",    # hyphen bullet
            "\u2018": "'",    # left single quote
            "\u2019": "'",    # right single quote
            "\u201a": "'",    # single low quote
            "\u201c": '"',    # left double quote
            "\u201d": '"',    # right double quote
            "\u201e": '"',    # double low quote
            "\u2026": "...",  # ellipsis
            "\u00a0": " ",    # non-breaking space
            "\u00b7": "-",    # middle dot
            "\u2015": "-",    # horizontal bar
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)

        # Final safety pass — replace remaining non-latin-1 chars
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    except Exception as e:
        logger.warning(
            f"Gemini call failed in {agent_name}: {str(e)[:100]}"
        )
        print(f"   ⚠️  Gemini call failed in {agent_name}: {e}")
        return f"[Gemini error in {agent_name}: {str(e)}]"


print("✅ Utility functions defined")
print("   log_event()        — timestamped pipeline logging")
print("   get_gemini_model() — safe model initialisation")
print("   safe_call_gemini() — error handled API call with font cleaning")


In [ ]:
def insight_agent(state: InsightForgeState) -> dict:
    """
    Agent 5 — Insight Agent
    -----------------------
    Reads  : state["cleaned_df"], state["schema_info"],
             state["eda_results"], state["cleaning_report"]
    Writes : state["insights"]
    """
    agent_name = "Insight Agent"
    print(f"\n{'─' * 55}")
    print(f"🔴 {agent_name} starting...")

    df     = state["cleaned_df"]
    schema = state["schema_info"]
    eda    = state["eda_results"]
    clean  = state["cleaning_report"]
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")
    target = schema.get("target_variable", "")

    logger.info(
        f"{agent_name} started — "
        f"sending to {GEMINI_MODEL}"
    )

    try:
        corr_lines = []
        if target and target in eda.get("correlation", {}):
            target_corr = {
                k: v for k, v in eda["correlation"][target].items()
                if k != target
            }
            sorted_corr = sorted(
                target_corr.items(),
                key=lambda x: abs(x[1]), reverse=True
            )
            corr_lines = [
                f"  {col}: {val:+.4f}"
                for col, val in sorted_corr[:6]
            ]

        vc_lines = []
        for col, counts in eda.get("value_counts", {}).items():
            vc_lines.append(f"\n  {col}:")
            for val, count in list(counts.items())[:5]:
                pct = round(count / eda["shape"][0] * 100, 1)
                vc_lines.append(f"    {str(val)}: {count} ({pct}%)")

        filled_summary = ", ".join([
            f"{col} ({info['count']} via {info['strategy']})"
            for col, info in clean.get("nulls_filled", {}).items()
        ])

        prompt = f"""
You are a senior data scientist and business analyst.
Analyse this dataset and provide a professional report.

DATASET CONTEXT
Domain   : {schema.get("domain",   "Unknown")}
Industry : {schema.get("industry", "Unknown")}
Summary  : {schema.get("summary",  "Unknown")}
Shape    : {eda["shape"][0]} rows x {eda["shape"][1]} columns
Target   : {target if target else "Not specified"}

DATA QUALITY ACTIONS TAKEN
Rows before cleaning   : {clean.get("rows_before", "N/A")}
Rows after cleaning    : {clean.get("rows_after",  "N/A")}
Duplicate rows removed : {clean.get("duplicates_removed", 0)}
Columns dropped        : {clean.get("columns_dropped", [])}
Null values filled     : {filled_summary or "None"}

STATISTICAL SUMMARY
{df.describe().round(2).to_string()}

CORRELATIONS WITH TARGET ({target.upper()})
{chr(10).join(corr_lines) if corr_lines else "No target variable specified"}

VALUE DISTRIBUTIONS
{chr(10).join(vc_lines)}

SKEWNESS
{chr(10).join([f"  {k}: {v}" for k, v in eda.get("skewness", {}).items()])}

SAMPLE DATA (first 5 rows)
{df.head(5).to_string()}

YOUR TASK
Write a professional analysis with EXACTLY these 6 sections.
Use actual numbers. No generic statements.

1. EXECUTIVE SUMMARY (3-4 sentences)
2. KEY FINDINGS (exactly 5 bullet points with numbers)
3. BUSINESS INSIGHTS (3-4 actionable recommendations)
4. DATA QUALITY ASSESSMENT
5. PATTERNS AND ANOMALIES
6. RECOMMENDED NEXT STEPS (3 specific actions)
"""

        print(f"   Prompt length  : {len(prompt):,} characters")
        print(f"   Sending to {GEMINI_MODEL}...")

        insights = safe_call_gemini(prompt, agent_name)

        log = log_event(
            state, agent_name,
            f"done — {len(insights):,} characters returned"
        )

        logger.info(
            f"{agent_name} complete — "
            f"{len(insights):,} characters returned"
        )

        print(f"   Response : {len(insights):,} characters")
        print(f"✅ {agent_name} complete")

        return {
            "insights"     : insights,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        msg = f"{agent_name} failed: {str(e)}"
        logger.error(f"{agent_name} FAILED — {str(e)}")
        print(f"   ❌ {msg}")
        return {
            "insights"     : "",
            "pipeline_log" : log_event(state, agent_name, f"FAILED — {e}"),
            "errors"       : errors + [msg]
        }

print("✅ insight_agent() defined")
